In [0]:
pip install findspark

In [0]:
import findspark, pyspark
from pyspark.sql import SparkSession
findspark.init()
spark = SparkSession.builder.appName('random_forest').getOrCreate()

In [0]:
from pyspark.ml.feature import RFormula

In [0]:
abs_path = "/Volumes/workspace/ml_with_spark/raw_data/Carros.csv"

carros = spark.read.csv(abs_path, header=True, inferSchema=True, sep=';')
carros.show(5)

In [0]:
rformula = RFormula(formula='HP ~ Consumo + Cilindros + Cilindradas', featuresCol='independente', labelCol='dependente')
carros_rf = rformula.fit(carros).transform(carros)
carros_rf.select('independente', 'dependente').show(5, truncate=False)

In [0]:
from pyspark.ml.feature import Normalizer

normalizador = Normalizer(inputCol='independente', outputCol='independente_norm', p=1.0)
carros_rf_norm = normalizador.transform(carros_rf)

In [0]:
carros_treino, carros_teste = carros_rf_norm.randomSplit([0.8, 0.2])
print(f'Quantidade de linhas no conjunto de treino: {carros_treino.count()}')
print(f'Quantidade de linhas no conjunto de teste: {carros_teste.count()}')


In [0]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
        featuresCol='independente', labelCol='dependente', numTrees=500, maxDepth=10, seed=20
    )
modelo = rf.fit(carros_treino)


In [0]:
previsao = modelo.transform(carros_teste)
previsao.select('dependente', 'prediction').show()

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluador = RegressionEvaluator(
        labelCol='dependente',
        predictionCol='prediction',
        metricName='rmse'
    )

rmse = evaluador.evaluate(previsao)
print(f'RMSE: {rmse}')